# Athena 02 — Register Yelp Reviews CSV as an Athena Table

Creates an external Athena table over the raw-reviews CSV that notebook `01_setup_S3_bucket.ipynb` uploaded to `s3://<bucket>/raw/reviews/`.

In [ ]:
import boto3
import pandas as pd
from pyathena import connect

%store -r bucket
%store -r region
%store -r database_name
%store -r s3_staging_dir
%store -r raw_reviews_prefix

table_name = "reviews_raw"
s3_reviews_location = f"s3://{bucket}/{raw_reviews_prefix}/"

print("Database:        ", database_name)
print("Table to create: ", table_name)
print("S3 location:     ", s3_reviews_location)

%store table_name
%store s3_reviews_location

conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

In [ ]:
drop_statement = f"DROP TABLE IF EXISTS {database_name}.{table_name}"
print(drop_statement)
pd.read_sql(drop_statement, conn)

In [ ]:
create_statement = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.{table_name} (
    review_id   STRING,
    business_id STRING,
    user_id     STRING,
    stars       INT,
    review_text STRING,
    date        STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    'separatorChar' = ',',
    'quoteChar'     = '\\"',
    'escapeChar'    = '\\\\'
)
LOCATION '{s3_reviews_location}'
TBLPROPERTIES ('skip.header.line.count'='1')
"""
print(create_statement)
pd.read_sql(create_statement, conn)

In [ ]:
df_show = pd.read_sql(f"SHOW TABLES IN {database_name}", conn)
df_show

## Run a sample query to confirm registration

In [ ]:
sample_df = pd.read_sql(
    f"SELECT review_id, stars, SUBSTR(review_text, 1, 80) AS review_snippet FROM {database_name}.{table_name} LIMIT 5",
    conn,
)
sample_df

In [ ]:
count_df = pd.read_sql(
    f"SELECT COUNT(*) AS row_count FROM {database_name}.{table_name}",
    conn,
)
count_df

In [ ]:
ingest_create_athena_table_passed = table_name in df_show.values
%store ingest_create_athena_table_passed
print("Athena reviews_raw table registered:", ingest_create_athena_table_passed)

## Done

Continue to `03_Convert_CSV_To_Parquet_With_Athena.ipynb` to materialize a Parquet copy of the data for faster downstream queries.